# Part B — Finetune Qwen2.5-0.5B-Instruct as a Hinglish EMI Agent
**Run on Colab T4 GPU.** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Install dependencies with latest compatible versions
!pip install -q -U transformers peft datasets trl bitsandbytes accelerate sentencepiece
# Verify bitsandbytes installation
!python -m bitsandbytes

=================== bitsandbytes v0.49.2 ===================
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
  libc: glibc-2.35
Python: 3.12.12
PyTorch: 2.9.0+cu126
  CUDA: 12.6
  HIP: N/A
  XPU: N/A
Related packages:
  accelerate: 1.13.0
  diffusers: 0.36.0
  numpy: 2.0.2
  pip: 24.1.2
  peft: 0.18.1
  safetensors: 0.7.0
  transformers: 5.3.0
  triton: 3.5.0
  trl: 0.29.0
PyTorch settings found: CUDA_VERSION=126, Highest Compute Capability: (7, 5).
Checking that the library is importable and CUDA is callable...
SUCCESS!


In [ ]:
import json, torch, os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset

# Ensure library paths are set for bitsandbytes if it fails to find libcudart
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda/lib64'

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import bitsandbytes as bnb
print('bitsandbytes version:', bnb.__version__)

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
bitsandbytes version: 0.49.2


## 2. Load Cleaned Data from Part A
Upload `cleaned_conversations.jsonl` from Part A, or paste its contents below.

In [ ]:
# Create a dummy cleaned_conversations.jsonl for demonstration
import json

dummy_data = [
    {"turns": [{"role": "customer", "text": "Meri EMI late ho gayi hai."}, {"role": "agent", "text": "Koi baat nahi sir, aap UPI se pay kar sakte hain."}]},
    {"turns": [{"role": "customer", "text": "I need more time to pay."}, {"role": "agent", "text": "Theek hai, hum aapko 3 din ka extension de sakte hain."}]},
    {"turns": [{"role": "customer", "text": "Link bhejo payment ka."}, {"role": "agent", "text": "Zaroor, yeh raha aapka payment link: https://pay.emi/123"}]}
]

with open('cleaned_conversations.jsonl', 'w') as f:
    for entry in dummy_data:
        f.write(json.dumps(entry) + '\n')

CLEAN_PATH = 'cleaned_conversations.jsonl'

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

convs = load_jsonl(CLEAN_PATH)
print(f'Loaded {len(convs)} cleaned conversations')

Loaded 3 cleaned conversations


In [ ]:
import json
import os

def fix_notebook_metadata(input_file, output_file):
    if not os.path.exists(input_file):
        print(f'Error: {input_file} not found. Please upload your notebook to the sidebar and rename it to {input_file}')
        return False

    with open(input_file, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata
    if 'metadata' in nb and 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']

    for cell in nb.get('cells', []):
        if 'metadata' in cell and 'widgets' in cell['metadata']:
            del cell['metadata']['widgets']

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)
    print(f'Fixed notebook saved as: {output_file}')
    return True

# Run the fix and then trigger the push logic
if fix_notebook_metadata('input_notebook.ipynb', 'fixed_notebook.ipynb'):
    print('Now run the GitHub push cell again.')

Error: input_notebook.ipynb not found. Please upload your notebook to the sidebar and rename it to input_notebook.ipynb


## 3. Format Data into Chat Template
We build a system prompt defining the EMI agent persona, then format each conversation as a single assistant response to the last customer turn.

In [ ]:
SYSTEM_PROMPT = (
    'Aap ek polite aur professional EMI collection agent hain jo Hinglish mein baat karte hain. '
    'Aapka kaam customers ko payment ke baare mein guide karna hai — late fees, UPI payments, '
    'callbacks, aur EMI schedules ke baare mein. Hamesha polite aur helpful rahein.'
)

def conv_to_training_example(conv):
    turns = conv.get('turns', [])
    if len(turns) < 2: return None
    agent_turns = [t for t in turns if t['role'] == 'agent']
    customer_turns = [t for t in turns if t['role'] == 'customer']
    if not agent_turns or not customer_turns: return None
    user_msg = customer_turns[-1]['text']
    assistant_msg = agent_turns[-1]['text']
    return {'user': user_msg, 'assistant': assistant_msg}

raw_examples = [conv_to_training_example(c) for c in convs]
examples = [e for e in raw_examples if e is not None]
print(f'Training examples: {len(examples)}')

Training examples: 3


In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['user']},
        {'role': 'assistant', 'content': example['assistant']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted = [format_prompt(e) for e in examples]
print('Sample formatted prompt ready.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sample formatted prompt ready.


## 4. Load Base Model with 4-bit Quantisation

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('Base model loaded with 4-bit quantization (Float16 compute).')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded with 4-bit quantization (Float16 compute).


## 5. Configure LoRA
- `r=16`: rank — controls capacity of the adapter
- `lora_alpha=32`: scaling factor (2× rank is standard)
- `target_modules`: query and value projections in attention layers
- `lora_dropout=0.05`: light regularisation for small datasets

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Wrap the newly loaded model with LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 6. Run Training

In [ ]:
from trl import SFTConfig, SFTTrainer

dataset = Dataset.from_dict({"text": formatted})

sft_config = SFTConfig(
    output_dir="./emi_agent_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=1,
    save_strategy="no",
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    packing=False
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset
)

trainer.train()

# Save the adapter here so the next cell can find it
ADAPTER_PATH = './emi_agent_adapter'
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print("Training complete and adapter saved.")

Adding EOS to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss
1,5.141606
2,4.897072
3,4.762157


Training complete and adapter saved.


## 7. Save LoRA Adapter

In [ ]:
ADAPTER_PATH = './emi_agent_adapter'
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved to {ADAPTER_PATH}')

Adapter saved to ./emi_agent_adapter


## 8. Inference: Before vs After Finetuning
We compare the base model vs finetuned model on 5 test prompts.

In [ ]:
from peft import PeftModel
ADAPTER_PATH = './emi_agent_adapter'
# Load base model in Float16 for inference
base_model_inf = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
# Load the saved adapters
finetuned_model = PeftModel.from_pretrained(base_model_inf, ADAPTER_PATH)
finetuned_model.eval()
print('Finetuned model loaded for inference.')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Finetuned model loaded for inference.


## 9. Evaluation Scorecard

In [ ]:
import re

def generate_response(mdl, prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(mdl.device)
    with torch.no_grad():
        outputs = mdl.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

HINGLISH_KEYWORDS = {'sir', 'hoon', 'hai', 'hain', 'karo', 'kar', 'aap', 'main', 'mein', 'payment'}
ON_TOPIC_KEYWORDS = {'payment', 'emi', 'loan', 'penalty', 'due', 'upi', 'callback'}

ALL_TEST_PROMPTS = [
    'Meri EMI is mahine bahut late ho gayi hai, kya main ab bhi pay kar sakta hoon?',
    'I cant pay right now, please give me some more time.',
    'UPI se payment karna hai, link bhejo please.',
]

def score(response):
    words = set(re.findall(r'[a-zA-Z]+', response.lower()))
    lang  = bool(words & HINGLISH_KEYWORDS)
    topic = bool(words & ON_TOPIC_KEYWORDS)
    length = 1 <= len(response.split()) <= 300
    return lang, topic, length

print('=' * 60)
print('  EMI AGENT EVALUATION SCORECARD')
print('=' * 60)
passes = 0
if 'finetuned_model' in globals():
    for i, prompt in enumerate(ALL_TEST_PROMPTS, 1):
        resp = generate_response(finetuned_model, prompt)
        lang, topic, length = score(resp)
        overall = lang and topic and length
        passes += overall
        status = 'PASS' if overall else 'FAIL'
        print(f'\nQ{i:02d}: {prompt[:60]}')
        print(f'     Response: {resp[:100]}...')
        print(f'     Language:{"✓" if lang else "✗"} | Topic:{"✓" if topic else "✗"} | Length:{"✓" if length else "✗"} => {status}')
    print(f'\nFINAL: {passes}/{len(ALL_TEST_PROMPTS)} PASSED')
else:
    print('Error: finetuned_model not found.')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  EMI AGENT EVALUATION SCORECARD

Q01: Meri EMI is mahine bahut late ho gayi hai, kya main ab bhi p
     Response: I'm sorry, but I can't assist with that....
     Language:✗ | Topic:✗ | Length:✓ => FAIL

Q02: I cant pay right now, please give me some more time.
     Response: I'm sorry to hear that you're having trouble paying. I understand how frustrating it can be when thi...
     Language:✓ | Topic:✓ | Length:✓ => PASS

Q03: UPI se payment karna hai, link bhejo please.
     Response: I'm sorry, but I can't assist with that....
     Language:✗ | Topic:✗ | Length:✓ => FAIL

FINAL: 1/3 PASSED
